# Data Cleaning Notebook

This notebook loads the raw Spotify dataset, inspects messy values, cleans numeric columns, encodes categorical features, and exports a cleaned dataset for modeling.

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path('../data/spotify-2023.csv')
OUT_PATH = Path('../data/spotify-2023-cleaned.csv')

print(f"Loading data from {DATA_PATH}")
df = pd.read_csv(DATA_PATH, encoding='latin-1')
print(df.head())
print(f"\nRaw shape: {df.shape}")


Loading data from ..\data\spotify-2023.csv
                            track_name    artist(s)_name  artist_count  \
0  Seven (feat. Latto) (Explicit Ver.)  Latto, Jung Kook             2   
1                                 LALA       Myke Towers             1   
2                              vampire    Olivia Rodrigo             1   
3                         Cruel Summer      Taylor Swift             1   
4                       WHERE SHE GOES         Bad Bunny             1   

   released_year  released_month  released_day  in_spotify_playlists  \
0           2023               7            14                   553   
1           2023               3            23                  1474   
2           2023               6            30                  1397   
3           2019               8            23                  7858   
4           2023               5            18                  3133   

   in_spotify_charts    streams  in_apple_playlists  ...  bpm key   mode  \
0  

In [13]:
pd.to_numeric(df['in_shazam_charts'], errors='coerce').sum()

np.float64(45854.0)

In [2]:
# Basic inspection of the raw data
print("Columns:")
print(df.columns.tolist())
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False).head(20))
print("\nDtypes:")
print(df.dtypes.head(20))


Columns:
['track_name', 'artist(s)_name', 'artist_count', 'released_year', 'released_month', 'released_day', 'in_spotify_playlists', 'in_spotify_charts', 'streams', 'in_apple_playlists', 'in_apple_charts', 'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts', 'bpm', 'key', 'mode', 'danceability_%', 'valence_%', 'energy_%', 'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%']

Missing values per column:
key                     95
in_shazam_charts        50
track_name               0
artist(s)_name           0
liveness_%               0
instrumentalness_%       0
acousticness_%           0
energy_%                 0
valence_%                0
danceability_%           0
mode                     0
bpm                      0
in_deezer_charts         0
in_deezer_playlists      0
in_apple_charts          0
in_apple_playlists       0
streams                  0
in_spotify_charts        0
in_spotify_playlists     0
released_day             0
dtype: int64

Dtypes:
track

In [6]:
# Clean numeric-looking columns that may contain strings or missing values
numeric_candidates = ['streams', 'in_spotify_charts', 'in_apple_charts', 'in_deezer_charts', 'in_deezer_playlists', 'in_shazam_charts', 'in_apple_playlists', 'in_spotify_playlists', 'bpm', 'danceability_%', 'energy_%', 'valence_%', 'acousticness_%', 'liveness_%', 'speechiness_%', 'artist_count', 'released_month']

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill remaining missing values with means for numeric columns
for col in numeric_candidates:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mean())

print("Numeric columns cleaned.")
print(df[numeric_candidates].dtypes)


Numeric columns cleaned.
streams                 float64
in_spotify_charts         int64
in_apple_charts           int64
in_deezer_charts          int64
in_deezer_playlists     float64
in_shazam_charts        float64
in_apple_playlists        int64
in_spotify_playlists      int64
bpm                       int64
danceability_%            int64
energy_%                  int64
valence_%                 int64
acousticness_%            int64
liveness_%                int64
speechiness_%             int64
artist_count              int64
released_month            int64
dtype: object


In [7]:
# Standardize obvious object columns and preserve useful categorical labels
for col in ['artist(s)_name', 'track_name', 'released_year', 'released_month', 'key', 'mode', 'artist_count']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col].replace({'nan': np.nan, 'None': np.nan}, inplace=True)

# Make sure categorical text is easy to work with downstream
for col in ['artist(s)_name', 'track_name', 'key', 'mode']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

print("Categorical cleanup complete.")


Categorical cleanup complete.


C:\Users\DanieHKombe\AppData\Local\Temp\ipykernel_11476\1150220879.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace({'nan': np.nan, 'None': np.nan}, inplace=True)
C:\Users\DanieHKombe\AppData\Local\Temp\ipykernel_11476\1150220879.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always b

In [8]:
# Final quality check and export
print(df.isna().sum().sort_values(ascending=False).head(10))
print(f"\nCleaned shape: {df.shape}")
df.to_csv(OUT_PATH, index=False)
print(f"Saved cleaned data to {OUT_PATH}")


track_name            0
artist(s)_name        0
liveness_%            0
instrumentalness_%    0
acousticness_%        0
energy_%              0
valence_%             0
danceability_%        0
mode                  0
key                   0
dtype: int64

Cleaned shape: (953, 24)
Saved cleaned data to ..\data\spotify-2023-cleaned.csv
